# 14b · 从测序读数到适应度（Nextflow + DiMSum）

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F]
                 ├─ 14a 稳定性换成 kcal/mol ✅
                 ├─ 14b 从测序读数到适应度          ← 你在这里
                 ├─ 14c 一个数拆成两个能量
                 ├─ 14d 少量数据 + 主动学习
                 ├─ 14e 位点地图
                 ├─ 14f 药理伴侣
                 └─ 14g 门面和讲法
```

## 背景：为什么要往上游走一步

到现在为止，项目里所有的标签——ProteinGym、MaveDB、Domainome、14a 的 Tsuboyama——都是**别人算好的数字**。我们拿到的是一张表：突变、分数。

但这张表是怎么来的？实验室做完实验，拿到的是测序仪吐出来的**几百万条短序列**（读段）。从读段到那张表，中间要经过：质控、去接头、拼接、计数、建误差模型、算适应度。**这一段就是药企生信科学家每天的工作**，也是面试必问的：

- "FASTQ 文件里有什么？Phred 质量值是什么意思？"
- "FastQC 报了红灯，你怎么判断要不要管？"
- "你的流程怎么保证别人能复现？"
- "Nextflow 和 Snakemake 你用过哪个？为什么要容器？"

这一章拿一个**真实发表的实验**，从原始读段开始，用工业标准的工具链跑一遍，最后和作者公布的结果**逐个变体对账**。

## 选了哪个实验

**Faure 等 2022，《Nature》**：Lehner 实验室测了 PSD95 蛋白第 3 个 PDZ 结构域（下面简称 PDZ3）的所有单突变和一部分双突变，测了两样东西：

| 实验 | 测的是什么 | 和我们哪章有关 |
|---|---|---|
| **AbundancePCA** | 细胞里还剩多少**折好的** PDZ3 | 14a 的稳定性 |
| **BindingPCA** | PDZ3 还能不能**结合**它的配体（一段叫 CRIPT 的肽） | 14c 的功能能 |

选它有三个理由：
1. **两种测量都有**，14c 要的"折叠能 + 功能能"正好从这里来。
2. **作者把一切都公开了**：原始读段在 SRA，DiMSum 的命令行和实验设计表在网盘，最终适应度在 GEO。这意味着可以做**精确复现**，而不是"大概差不多"。
3. **数据量适合笔记本**：12 个测序样本、约 2.6 GB。

## 这一章做什么

| 步骤 | 内容 |
|---|---|
| 1 | 实验在测什么：深度突变扫描、PCA、适应度的定义 |
| 2 | 进货：24 个 FASTQ + 作者的 2 张表进 S3 raw，按 ENA 的 MD5 核对 |
| 3 | FASTQ 长什么样：一条读段、质量值、双端测序 |
| 4 | Nextflow：为什么是它，流程怎么组织，为什么跑在 AWS Batch 上 |
| 5 | 运行：从笔记本到云，一路踩过的六个坑 |
| 6 | 质控：FastQC + MultiQC，哪些红灯可以不管 |
| 7 | DiMSum 在干什么 |
| 8 | **对账**：和作者公布的适应度逐个比 |
| 9 | 结果在哪：manifest 写明来历，给 14c 用 |

---

## 准备

运行下面这一格：连上 AWS，载入代码，检查 Nextflow 和 Docker 都在。**不用看代码。**

In [ ]:
import json, os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import boto3
import pandas as pd
from qafi_platform import ingest
from qafi_core import dms
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True, tail=None):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).rstrip().splitlines()
    print("\n".join(out[-tail:] if tail else out))
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
aws = boto3.Session(profile_name="qafi", region_name="eu-central-1")
s3, lam = aws.client("s3"), aws.client("lambda")
buckets = json.loads(subprocess.run("terraform -chdir=infra/dev output -json bucket_names",
                                    shell=True, capture_output=True, text=True, check=True).stdout)
RAW, ARTIFACTS = buckets["raw"], buckets["artifacts"]
READS = "datasets/psd95-pdz3-ddpca/GSE184042"           # 读段在 raw 里的位置
RESULTS = "pipelines/dms_fitness"                       # 流程结果在 artifacts 里的位置
QUEUE = subprocess.run("terraform -chdir=infra/dev output -raw batch_job_queue",
                       shell=True, capture_output=True, text=True, check=True).stdout
DMS = ROOT / "data/dms"                                 # 本机只放下载回来的小结果，不进 Git
DMS.mkdir(parents=True, exist_ok=True)
run("nextflow -version | grep version")
print("Batch 队列:", QUEUE)

### 结果怎么看
Nextflow 版本号，和 AWS Batch 队列的名字。这一章的计算**不在你的 Mac 上**：Mac 上的 Nextflow 只负责调度，每一步都作为一个任务提交到 AWS Batch，在云上的机器里跑（第 4、6 步讲为什么、怎么做）。

---

## 第 1 步 · 实验在测什么

### 在干什么
弄懂这批数据是怎么产生的。不懂实验，就看不懂流程里每一步在干嘛。

### 背景：深度突变扫描（DMS）

一句话：**造出一个蛋白的成千上万种突变体，让它们在同一个试管里竞争，然后用测序数一数谁赢了。**

```
           ┌────────── 选择（细胞生长 29 小时）──────────┐
 突变库     │                                             │
 ─────▶  [输入]  测序：每个突变体有多少条读段            [输出]  测序：选择之后每个还剩多少
          input                                          output
```

- 如果一个突变体在输出里**比例变高**了，说明它在选择中**占优**；比例变低，说明它**吃亏**。
- **适应度**（fitness）= 输出/输入的比值，取对数，再减去野生型的这个值：

```
fitness = log( 输出计数 / 输入计数 ) − log( 野生型输出 / 野生型输入 )
```

所以**野生型的适应度 = 0**，比野生型差的是负数。

### 选择用的是什么：PCA（蛋白片段互补）

这里的 PCA 不是"主成分分析"，是 **protein-fragment complementation assay**。原理：把酵母生长必需的一个酶（DHFR）切成两半。两半自己不工作，**被拉到一起**才工作。

| 实验 | 两半各接在哪 | 细胞长得好意味着 |
|---|---|---|
| **AbundancePCA** | 一半接在 PDZ3 上，另一半大量表达、到处都是 | PDZ3 **折好的量多**（折不好的会被细胞降解） |
| **BindingPCA** | 一半接在 PDZ3 上，另一半接在配体肽 CRIPT 上 | PDZ3 **能结合** CRIPT |

所以 AbundancePCA 读的是**稳定性**，BindingPCA 读的是**稳定性 × 结合**——这正是 14c "两个能量"那个物理读数层要拆开的东西。

### 突变库是怎么造的：切口诱变 + 9 个背景

作者用一种叫**切口诱变**（nicking mutagenesis）的方法，把 PDZ3 的每一个密码子都换成其他所有可能，得到全部单突变。

另外，他们事先挑了 **9 个"背景"突变**（比如第 13 位 CTG→CGT，即 L13R），在每个背景上**再做一遍**全部单突变。于是库里有：
- 野生型
- 全部单突变
- **设计好的双突变**：一个背景 + 另一个任意突变

为什么要双突变？因为 14c 要从**两个突变的相互作用**里把折叠能和结合能分开——只有单突变的话，这两个量纠缠在一起解不开。

**后面第 7 步会用到这件事**：测序里会出现很多"不是设计出来的"双突变（测序或合成错误造成的）。作者的做法是只保留至少有一个突变是 9 个背景之一的双突变，其他的扔掉。

---

## 第 2 步 · 进货：读段和标准答案

### 在干什么
把 24 个 FASTQ 文件（12 个样本 × 双端）和作者公布的 2 张适应度表，按平台规矩搬进 S3 raw。

### 背景：这批数据存在哪

测序数据有专门的公共仓库：

| 仓库 | 存什么 | 这批数据的编号 |
|---|---|---|
| **GEO**（NCBI） | 一个研究的"总目录"和处理后的结果 | GSE184042 |
| **SRA**（NCBI） / **ENA**（欧洲） | 原始读段，两边互相镜像 | PRJNA762874 |

我们从 **ENA** 下载，因为它直接提供压缩好的 `.fastq.gz` 和每个文件的 **MD5 校验值**（SRA 给的是它自己的 `.sra` 格式，还要转换）。

### 用了什么技术
和 03b、14a 一样：数据源写进 `config/sources.toml`，送货员 Lambda 从 ENA / GEO 直接搬进 S3，每个文件到位后验货员自动检查并写收货单。**文件不经过你的电脑**。

26 个文件一次全部发出，26 个 Lambda 同时搬，几分钟搞定。

In [ ]:
sources = [s for s in ingest.load_sources(ROOT / "config" / "sources.toml") if s["dataset"] == "psd95-pdz3-ddpca"]
print(len(sources), "个文件")
receipts = ingest.fetch_sources(lam, s3, RAW, sources)

### 结果怎么看
26 行，每行 `accepted` + 文件大小。

下面**和 ENA 公布的 MD5 逐个核对 S3 里的副本**。

### 为什么要核对 MD5
S3 的收货单已经证明"搬到 S3 的和 ENA 声明的大小一样"。但大小一样不代表内容一样。MD5 是文件内容的"指纹"：改动任何一个字节，指纹就完全不同。**用数据提供方公布的指纹核对**，是测序数据交接的标准动作——很多公司的数据接收流程里，指纹对不上的文件直接拒收。

文件不落到你的硬盘上：一边从 S3 读、一边算指纹（流式），2.6 GB 读一遍大约两三分钟。

In [ ]:
import hashlib
ena = pd.read_csv("https://www.ebi.ac.uk/ena/portal/api/filereport?accession=PRJNA762874"
                  "&result=read_run&fields=run_accession,fastq_md5&format=tsv", sep="\t")
expected = {f"{run}_{i + 1}.fastq.gz": m for run, md5s in zip(ena.run_accession, ena.fastq_md5)
            for i, m in enumerate(md5s.split(";"))}
def s3_md5(key):
    digest = hashlib.md5()
    for chunk in s3.get_object(Bucket=RAW, Key=key)["Body"].iter_chunks(8 << 20):
        digest.update(chunk)
    return digest.hexdigest()
fastqs = [s["key"] for s in sources if s["key"].endswith(".fastq.gz")]
bad = [k for k in fastqs if s3_md5(k) != expected[Path(k).name]]
print(f"{len(fastqs)} 个 FASTQ，S3 副本和 ENA 的 MD5 不一致的: {bad or '没有'}")

### 结果怎么看
24 个文件，**0 个不一致**。

---

## 第 3 步 · FASTQ 长什么样

### 在干什么
打开一个文件，看一条真实的读段，弄懂每一行是什么。

### 背景
FASTQ 是测序数据的通用格式。**每 4 行是一条读段**：

```
@SRR15872849.1 1/1      ← 第 1 行：名字
TCCTGGGAGGTGGAGCTAGC... ← 第 2 行：碱基序列
+                       ← 第 3 行：分隔符
11>>111111AAA1AEGD1A... ← 第 4 行：每个碱基的质量值，一个字符对应一个碱基
```

**质量值（Phred score）**：测序仪对每个碱基"有多大把握"。字符的 ASCII 码减 33 就是 Q 值：

| Q 值 | 读错的概率 | 字符举例 |
|---|---|---|
| 10 | 1/10 | `+` |
| 20 | 1/100 | `5` |
| 30 | 1/1,000 | `?` |
| 40 | 1/10,000 | `I` |

**双端测序**（paired-end）：同一个 DNA 片段，从两头各读一次。这批数据是 MiSeq 的 **2 × 250**：每头读 250 个碱基。PDZ3 的编码区 + 两头的固定序列大约 300 多个碱基，所以**两条读段在中间有很长一段重叠**。这很重要：重叠区每个位置被读了两次，可以互相纠错。

### 执行
看第一个样本的第一对读段。

In [ ]:
import gzip
from itertools import islice
for end in (1, 2):
    body = s3.get_object(Bucket=RAW, Key=f"{READS}/SRR15872849_{end}.fastq.gz")["Body"]
    with gzip.open(body, "rt") as f:                      # 只读开头几行，不下载整个文件
        name, seq, _, qual = [line.rstrip() for line in islice(f, 4)]
    print(f"读段 {end}: {name}\n  序列 {seq[:70]}...\n  质量 {qual[:70]}...")
    print(f"  前 10 个碱基的 Q 值: {[ord(c) - 33 for c in qual[:10]]}\n")

### 结果怎么看：这条读段里有什么

读段 1 的开头是 `TCCT` + **`GGGAGGTGGAGCTAGC`**——后者是 PDZ3 编码区前面的**固定序列**（作者在 DiMSum 命令里写的 `cutadapt5First`），之后才是 PDZ3 本身。

把这条读段的 PDZ3 部分和野生型逐个密码子对比，能读出它带了**两个突变**：

| 位置 | 野生型 | 这条读段 | 氨基酸 |
|---|---|---|---|
| 第 8 位 | CGG | CAG | R → Q |
| 第 13 位 | CTG | **CGT** | L → R |

第 13 位的 CGT 正是作者设计的 **9 个背景之一**（`13cgt`）。所以这条读段是一个**设计好的双突变**：背景 L13R + 单突变 R8Q。第 1 步讲的东西，在第一条数据里就看到了。

读段 2 的开头是 **`GCGTGACATAACTAATAAGCTT`**（`cutadapt5Second`），即另一头的固定序列，后面是 PDZ3 的**反向互补**——因为它是从另一头往回读的。

### 面试怎么说
> "FASTQ stores four lines per read: name, bases, a separator and per-base Phred qualities, Q = −10·log10(error probability), encoded as ASCII minus 33. For a 2×250 MiSeq amplicon shorter than 500 bp the pairs overlap, so they can be merged and each overlapping base is called twice."

---

## 第 4 步 · Nextflow：为什么是它，以及为什么跑在 AWS Batch 上

### 在干什么
弄懂我们用来串起所有步骤的工具、流程文件是怎么组织的，以及它在哪里执行。

### 背景：为什么不写一个 Python 脚本一口气跑完

生信流程有几个特殊要求，普通脚本都满足不了：

| 要求 | 为什么难 | Nextflow 怎么解决 |
|---|---|---|
| **每一步用不同的软件**，版本还得钉死 | FastQC 是 Java、DiMSum 是 R、cutadapt 是 Python | 每一步跑在自己的**容器**里，版本写在配置里 |
| **跑到一半挂了能接着跑** | 一个流程动辄几小时 | `-resume`：已经成功的步骤直接复用结果 |
| **能并行就并行** | 12 个样本的 FastQC 互不相关 | 自动按依赖关系排程 |
| **同一份代码，笔记本上和云上都能跑** | 数据量从 MB 到 TB | 换一个 `-profile` 就从本机切到 AWS Batch / Slurm |
| **出了问题能查** | 几十个步骤，哪步慢、哪步占内存 | 自动生成时间线、资源报告 |

**Nextflow** 是现在生物信息领域的**事实标准**，尤其在药企。**nf-core** 社区维护着 100 多个按统一规范写的 Nextflow 流程（RNA-seq、变异检测、单细胞……）。另一个常见选择是 **Snakemake**（Python 风格，学术界多一些）。招聘要求里写的最多的是 Nextflow。

### 我们的流程长什么样

```
pipelines/dms_fitness/
├── main.nf                    流程本身：5 个步骤和它们之间的连线
├── nextflow.config            每一步用哪个容器、几个 CPU、多少内存；两个 profile：docker（本机）、awsbatch（云）
├── bin/keep_designed_doubles.R  作者的过滤脚本（参数化后）
└── experiments/
    ├── psd95_pdz3_abundance.csv   样本表：哪个样本是输入/输出、第几个重复、对应哪两个 FASTQ
    ├── psd95_pdz3_abundance.yaml  这个实验的全部参数（照抄作者公布的 DiMSum 命令）
    └── psd95_pdz3_binding.*       另一个实验，同样两个文件
```

**代码和参数分开**：换一个实验只需要写一份新的 `.csv` 和 `.yaml`，流程本身一行不改。这是 nf-core 的规范。

五个步骤：

```
24 个 FASTQ ──┬──▶ FASTQC（每个样本一次）──▶ MULTIQC（汇总成一份报告）
              │
              └──▶ DIMSUM_COUNT ──▶ KEEP_DESIGNED_DOUBLES ──▶ DIMSUM_FITNESS ──▶ 适应度表
                  （第 1–4 阶段：      （作者的过滤：只留       （第 4–5 阶段：
                   去固定序列、拼接、    设计好的双突变）          误差模型、适应度）
                   计数）
```

每一步的软件都来自 **BioContainers**：bioconda 项目为每个生信软件的每个版本自动构建的 Docker 镜像。我们钉死的版本：

| 步骤 | 镜像 |
|---|---|
| FastQC | `fastqc:0.12.1` |
| MultiQC | `multiqc:1.33` |
| DiMSum | `r-dimsum:1.2.8` ← **和作者论文里用的版本完全一样** |

### 在哪里执行：AWS Batch

**AWS Batch** 是 AWS 的"排队跑任务"服务：你把任务（一个容器 + 要几个 CPU、多少内存）提交到一个**队列**，Batch 自己去开机器、跑完、没任务了就关机。**空闲时一分钱不花。** Nextflow 原生支持它：配置里写 `process.executor = 'awsbatch'`，每一步就自动变成一个 Batch 任务。

```
你的 Mac：nextflow run -profile awsbatch      ← 只负责调度，不做计算
         │ 提交任务
         ▼
AWS Batch 队列 qafi-dev-pipelines ──▶ 自动开 EC2 Spot 机器（x86，2 核 8 GB，100 GB 硬盘，最多 2 台）
         │                                │  每个任务：拉镜像 → 从 S3 取输入 → 跑 → 结果写回 S3
         ▼                                ▼
s3://raw/datasets/...（读段）    s3://artifacts/nextflow/work（中间文件，7 天后自动删）
                                 s3://artifacts/pipelines/...（结果）
```

| 新建的东西（`infra/dev/batch.tf`，Terraform） | 作用 |
|---|---|
| 计算环境 | 用哪种机器、最多几台、Spot（比按需便宜约 70%） |
| 任务队列 | Nextflow 往这里提交 |
| 机器的身份（IAM 角色） | 只能读 raw 的 `datasets/`、读写 artifacts 的 `nextflow/` 和 `pipelines/`、用加密密钥 |
| 启动模板 | 100 GB 硬盘；开机时用 conda 装好 AWS 命令行工具（Nextflow 把它挂进每个容器，用来收发 S3 文件） |
| 安全组 | 只出不进 |
| 生命周期规则 | `nextflow/work/` 里的中间文件 7 天后自动删 |

**为什么一开始在本机跑、后来搬上云**，第 5 步讲——那是一个真实的过程。

### CI 里也测了这个流程

`.github/workflows/ci.yml` 新加了一个任务：每次提交都跑 `nextflow lint`（静态检查），再跑一次 **stub 运行**——每一步不调用真正的软件，只执行一个"假动作"（创建空的输出文件），用几秒钟验证**步骤之间的连线**是对的，包括"流程生成的实验设计表有表头和 6 行"。nf-core 的流程都这样测。

### 面试怎么说
> "I wrapped DiMSum in a small Nextflow DSL2 pipeline: each step in a pinned BioContainers image, parameters per experiment in a YAML file, a stub run in CI that tests the wiring without data. It runs locally with Docker or on AWS Batch with a profile switch; the Batch side is Terraform — a Spot compute environment that scales to zero, a least-privilege instance role and an S3 work directory with a lifecycle rule."

---

## 第 5 步 · 运行：从笔记本电脑到云，一路踩过的六个坑

### 在干什么
跑完整的流程：两个实验，每个 12 个 FASTQ，在 AWS Batch 上。

### 执行
一个实验一条命令。参数全在 `.yaml` 里，命令本身很短：

```bash
nextflow run pipelines/dms_fitness -profile awsbatch -resume \
    -params-file pipelines/dms_fitness/experiments/psd95_pdz3_binding.yaml \
    --fastq_dir s3://<raw>/datasets/psd95-pdz3-ddpca/GSE184042 \
    --outdir   s3://<artifacts>/pipelines/dms_fitness/psd95_pdz3_binding \
    -work-dir  s3://<artifacts>/nextflow/work
```

- `-profile awsbatch`：每一步作为 Batch 任务跑
- `-resume`：已经成功的步骤不重跑（中间结果在 S3 的 work 目录里）
- 输入、中间文件、结果**全在 S3**；你的 Mac 只负责调度

Nextflow 用 AWS 的临时凭证提交任务。它的 SDK 不认识 `aws login` 的会话，所以 `~/.aws/config` 里加了一个专门的 profile `qafi-nextflow`：它用 `credential_process` 向 AWS 命令行要凭证，**快过期时自动换新的**（下面第 6 个坑讲为什么必须这样）：

In [ ]:
env = {**os.environ, "AWS_PROFILE": "qafi-nextflow", "AWS_REGION": "eu-central-1"}   # 会自动续期的凭证
for kind in ("binding", "abundance"):
    launch = DMS / "launch" / kind                       # 每个实验一个启动目录，互不干扰
    launch.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(
        f"nextflow -log {DMS}/{kind}.batch.log run {ROOT}/pipelines/dms_fitness -profile awsbatch -resume "
        f"-params-file {ROOT}/pipelines/dms_fitness/experiments/psd95_pdz3_{kind}.yaml "
        f"--samplesheet {ROOT}/pipelines/dms_fitness/experiments/psd95_pdz3_{kind}.csv "
        f"--fastq_dir s3://{RAW}/{READS} --outdir s3://{ARTIFACTS}/{RESULTS}/psd95_pdz3_{kind} "
        f"-work-dir s3://{ARTIFACTS}/nextflow/work -ansi-log false",
        shell=True, capture_output=True, text=True, env=env, cwd=launch)
    print(f"--- {kind}\n" + "\n".join((r.stdout + r.stderr).strip().splitlines()[-4:]))

### 结果怎么看
每个实验最后一行 `[SUCCESS] completed=…`。因为用了 `-resume`，如果之前跑过，这里几秒就结束，显示 `cached=…`。

两个实验加起来 20 个任务，大约 40 分钟（两台机器并行，Spot 价格下花费不到 0.1 美元）。

### 一路踩过的六个坑

这个流程**最开始是在你的 Mac 上跑的**（`-profile docker`），连续失败了好几次。每一次失败都是一个真实的、面试里能讲的故事。

**坑 1 · 内存不够，进程被系统杀掉。** 错误信息是：

```
Error in unserialize(node$con) : error reading from connection
```

这句话本身看不出原因（意思是"R 的一个并行子进程断了"）。真正的原因要去**系统日志**里找：

```bash
colima ssh -- sudo dmesg | grep "Killed process"
Memory cgroup out of memory: Killed process 70591 (R) ... anon-rss:1596736kB
```

**内核因为内存不够，把 R 的子进程杀掉了**（Linux 的 OOM killer）。规律：程序自己报的错往往很含糊，内存不足要看系统日志。查下来 DiMSum 每个子进程有两个内存高峰：切分文件时约 1.7 GB；过滤读段时约是**切块大小的 25 倍**（它把每个碱基的质量值展开成矩阵）。DiMSum 默认块大小是 3.5 GB——为集群设计的。解法：块改成 30 MB（`split_chunk_size`）、并行降到 2。**块大小不影响任何计数**，只是把同样的工作分成更多小份。

**坑 2 · 磁盘写满。** 内存问题解决后，又失败了：`No space left on device`。DiMSum 工作时会同时保留**解压版、切块版、去接头版**三份读段，一个实验要 25–30 GB 临时空间。加上失败的尝试留下的中间文件、内存吃紧时 macOS 写的交换文件，**你的 Mac 一度只剩 119 MB**。教训有两条：给长时间任务加**磁盘保护**（我们的运行脚本在空闲低于 8 GB 时自动停）；还有一个坑中坑——Docker 虚拟机里删掉的文件，宿主机上不会自动释放，要 `fstrim`。

**坑 3 · 结论：这活不该在笔记本上干。** 内存、磁盘、x86 模拟（Rosetta）三样都在拖后腿。这正是 Nextflow 的设计目的：**流程代码一行不改**，`-profile docker` 换成 `-profile awsbatch`，就搬到了云上。

**坑 4 · 云上也有看不见的上限，而且 Batch 不报错，只是不动。** 第一次在 Batch 上提交，14 个任务一直排队，一台机器都没开，计算环境的状态还是"健康"。最后在 **CloudTrail**（AWS 记录每一次 API 调用的日志）里找到了原因，一共两层：

| 限制 | 在哪看到的 | 影响 | 怎么办 |
|---|---|---|---|
| 新账号同时只能跑 **5 个 vCPU** 的 EC2 | Service Quotas 控制台 | 8 核的机器一台都开不出来 | 选小机器；或申请提高配额 |
| **Free plan 只能开免费套餐的机型** | CloudTrail：`not eligible for Free Tier` | 除了几种小机器，全部被拒 | 选其中最大的 **m7i-flex.large（2 核 8 GB）** |

2 核 8 GB 够用，因为第 1 个坑已经把 DiMSum 的内存压到了约 4 GB（2 个并行、30 MB 切块）。配额 5 个 vCPU 正好能开两台，两个实验并行跑。如果以后升级到付费计划、提高配额，只需要在 `batch.tf` 里把机型换大、`max_vcpus` 调高，流程本身不用动。

**通用教训：调度器安静地不干活时，先查配额和账号限制，再查代码。**

**坑 5 · 机器起来了，任务全部秒挂。** 错误是：

```
/opt/aws-cli/bin/aws: error while loading shared libraries: libz.so.1: cannot open shared object file
```

背景：Nextflow 在每个任务的容器里用 **AWS 命令行**从 S3 取输入、把结果写回去。命令行装在机器上，由 Nextflow **挂进容器**里用。AWS 官方的命令行安装包依赖机器系统里的一个压缩库（`libz`），而 BioContainers 的容器为了小，里面根本没有这个库——挂进去也跑不起来。

解法就是 Nextflow 官方文档的推荐写法：**用 conda 装 AWS 命令行**。conda 环境把它需要的每一个库都装在自己的文件夹里，整个文件夹挂进任何容器都能用。改的只是启动模板里的三行安装命令（`infra/dev/batch.tf`）。

这个坑的价值在于：它是"在我机器上能跑"和"在别人的容器里能跑"的区别——**依赖要么打包进来，要么就别指望环境里有。**

**坑 6 · 跑到一半，凭证过期了。** 修好命令行之后，FastQC 全部成功，DiMSum 跑着跑着，Nextflow 突然把所有任务都停了：

```
S3Exception: The provided token has expired.
```

原因：我最初是把 `aws login` 的临时凭证**导出成固定的环境变量**交给 Nextflow 的。临时凭证本来就只有短时间有效（这是安全设计），一个要跑几十分钟的流程撑不到最后。解法是公司里的标准做法：给这类工具一个 **`credential_process`** profile——SDK 发现凭证快过期，就自己调用这条命令拿新的。重跑时加了 `-resume`，已经成功的 FastQC 和 MultiQC 直接从 S3 的缓存里复用，只补跑 DiMSum。

这也是 `-resume` 的价值最直观的一次：**失败不用从头来。**

### 面试怎么说
> "On the laptop the pipeline hit the OOM killer, then filled the disk — DiMSum keeps three copies of the reads. Rather than tune around a laptop, I moved it to AWS Batch with a profile switch. The first submission sat in RUNNABLE with a healthy compute environment; CloudTrail showed why — a 5-vCPU EC2 quota and Free-plan instance-type restrictions — so I sized it to two 2-vCPU, 8 GB Spot machines, which the earlier memory profiling said was enough. The lesson I'd generalise: when a scheduler silently doesn't scale, check quotas before the code."

---

## 第 6 步 · 质控：哪些红灯可以不管

### 在干什么
看 FastQC 的结果，判断数据能不能用。

### 背景
**FastQC** 对每个 FASTQ 文件做 11 项检查，每项给绿 / 黄 / 红灯。**MultiQC** 把所有样本的 FastQC 结果汇总成一页。这两个是所有测序项目的第一步。

**但 FastQC 的标准是按"全基因组测序"定的**——那种数据是从整个基因组随机打断来的，序列五花八门。我们的数据是**扩增子**：几百万条读段都是同一段 PDZ3 基因，只差一两个碱基。用基因组的标准看扩增子，有几项一定会亮红灯。

**会不会判断哪些红灯可以不管，是面试官最想听的。**

### 执行
从 S3 取回 MultiQC 的汇总表（几十 KB）：

In [ ]:
def fetch(prefix, local):            # 把 artifacts 里 prefix 下的文件取到本机（local 是文件夹）
    local.mkdir(parents=True, exist_ok=True)
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=ARTIFACTS, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("/"):
                continue                 # S3 控制台留下的"文件夹"标记，0 字节，不是文件
            target = local / obj["Key"][len(prefix):].lstrip("/") if obj["Key"] != prefix else local / Path(prefix).name
            target.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(ARTIFACTS, obj["Key"], str(target))
for kind in ("binding", "abundance"):
    fetch(f"{RESULTS}/psd95_pdz3_{kind}/qc/psd95_pdz3_{kind}_multiqc_report_data", DMS / f"results/{kind}/qc/data")
    fetch(f"{RESULTS}/psd95_pdz3_{kind}/counts/filter_summary.txt", DMS / f"results/{kind}/counts")
qc = {kind: pd.read_csv(DMS / f"results/{kind}/qc/data/multiqc_fastqc.txt", sep="\t").set_index("Sample")
      for kind in ("binding", "abundance")}
checks = ["per_base_sequence_quality", "per_base_n_content", "adapter_content", "sequence_length_distribution",
          "per_base_sequence_content", "per_sequence_gc_content", "sequence_duplication_levels",
          "overrepresented_sequences"]
both = pd.concat(qc.values())
summary = pd.DataFrame({c: both[c].value_counts() for c in checks}).fillna(0).astype(int).T
print(f"{len(both)} 个 FASTQ 文件 · 读段数 {both['Total Sequences'].min():,.0f}–{both['Total Sequences'].max():,.0f}")
summary

### 结果怎么看

| 检查项 | 结果 | 要不要管 | 为什么 |
|---|---|---|---|
| 每个位置的测序质量 | 全部 pass | **要管，过了** | 这是最重要的一项 |
| N 的比例（读不出的碱基） | 全部 pass | **要管，过了** | |
| 接头污染 | 全部 pass | **要管，过了** | 读段长度 250 < 片段长度，读不到接头 |
| 长度分布 | 全部 pass | | MiSeq 每条都是 251 |
| 每个位置的碱基组成 | **全部 fail** | **不用管** | 所有读段是同一段序列，第 20 位就是 C，不可能 A/C/G/T 各 25% |
| GC 含量分布 | **全部 fail** | **不用管** | 同理，所有读段 GC 含量几乎一样，分布是一根尖峰而不是钟形 |
| 重复读段 | **全部 fail** | **不用管** | 只剩 4–5% 是"不重复的"——扩增子文库里同一个变体本来就有几百条读段，这正是我们要数的东西 |
| 过度代表的序列 | **全部 fail** | **不用管** | 最常见的序列就是野生型 |

**结论：数据质量没问题。** 4 项红灯全是扩增子文库的固有特征，不是数据坏了。

再看测序质量沿读段怎么变（MiSeq 的典型情况）：

In [ ]:
import ast
raw = pd.read_csv(DMS / "results/binding/qc/data/fastqc_per_base_sequence_quality_plot.txt", sep="\t").set_index("Sample")
curve = {s: dict(ast.literal_eval(v) for v in row.dropna()) for s, row in raw.iterrows()}
at = lambda q, k: q[min(q, key=lambda x: abs(x - k))]
pd.DataFrame({s: {f"第 {k} 位": round(at(q, k), 1) for k in (2, 100, 200, 245, 251)} for s, q in curve.items()}).T.head(4)

### 结果怎么看

- **读段 1**（`_1`）：从头到尾都在 **Q33 以上**，很好。
- **读段 2**（`_2`）：末端掉到 **Q26–29**。这是 MiSeq 的常态，越往后化学反应越不稳定。

这就是为什么 DiMSum 要把两条读段**拼起来**：读段 2 末端质量差的那一段，正好和读段 1 开头质量好的那一段重叠，拼接时取质量高的那个碱基。拼完以后还按"预期错误数"过滤：作者设的是一条拼好的读段**整条加起来最多 0.25 个预期错误**（`vsearch_maxee: 0.25`），比通常的 1.0 严得多——因为我们要区分只差一个碱基的突变体，一个测序错误就会把野生型读成突变体。

### 面试怎么说
> "FastQC thresholds assume shotgun libraries. For an amplicon DMS library, failing per-base content, GC distribution, duplication and overrepresented sequences is expected; what I check is per-base quality, N content and adapters. Read 2 decays to about Q27 at the end, which the pair merging and a strict max-expected-errors filter take care of."

---

## 第 7 步 · DiMSum 在干什么

### 在干什么
弄懂这个工具内部的五个阶段，以及作者为什么要分两次跑。

### 背景

**DiMSum**（Faure 等 2020，*Genome Biology*）是 Lehner 实验室为深度突变扫描写的流程，分两大部分：

**WRAP：读段 → 计数**

| 阶段 | 做什么 | 用的工具 | 参数（照抄作者） |
|---|---|---|---|
| 1 | 质控 | FastQC | |
| 2 | 去掉两头的固定序列 | **cutadapt** | `cutadapt5First/Second`、最少剩 200 个碱基、允许 20% 错配 |
| 3 | 拼接双端读段 + 质量过滤 | **VSEARCH** | 碱基质量 ≥ Q25、整条预期错误 ≤ 0.25 |
| 3 | 数每种序列出现了几次 | **starcode** | |

**STEAM：计数 → 适应度**

| 阶段 | 做什么 |
|---|---|
| 4 | 合并样本，去掉插入缺失，把 DNA 序列翻译成蛋白序列 |
| 5 | 去掉输入计数太少的变体 → 按重复实验校准尺度 → **拟合误差模型** → 算适应度和它的标准误 |

**误差模型**是 DiMSum 的核心贡献：一个突变体在输入里只有 10 条读段和有 1,000 条读段，算出来的适应度可信度完全不同。DiMSum 用三个重复实验之间的离散程度，估计每个变体适应度的**标准误**（`sigma`）。14c 拟合能量时会用它来加权。

### 为什么要分两次跑

作者的命令是这样的：

```
第一次：DiMSum 阶段 1–4（计数）
         ↓
     过滤：只保留设计好的双突变（第 1 步讲的 9 个背景）
         ↓
第二次：DiMSum 从阶段 4 开始，读过滤后的计数，算适应度
```

中间那个过滤是**这个突变库特有的**：切口诱变库里，凡是"两个突变都不是背景"的双突变，都不是设计出来的，是合成或测序错误。它们计数很少、不可靠，留着会污染误差模型。

我们的流程原样复现了这个结构（`DIMSUM_COUNT → KEEP_DESIGNED_DOUBLES → DIMSUM_FITNESS`），过滤脚本就是作者的 R 脚本，只是把写死的路径和背景列表改成了参数。

In [ ]:
for kind in ("binding", "abundance"):
    print(f"{kind:10s}", (DMS / f"results/{kind}/counts/filter_summary.txt").read_text().strip())

### 结果怎么看
binding 从 95,315 个变体筛到 25,581 个，abundance 从 83,137 筛到 24,493——**差不多 70% 的双突变被丢掉了**。它们都是"两个突变都不在那 9 个背景里"的组合：不是设计出来的，是合成或测序错误的产物，计数很少、不可靠。留着会污染误差模型。

---

## 第 8 步 · 对账：和作者公布的适应度逐个比

### 在干什么
这是这一章的考试：我们从原始读段算出来的适应度，和作者 2022 年发在 GEO 上的那张表，**按蛋白序列一一对齐**，看对不对得上。

### 背景：对账的标准

- **变体数**：作者表里的每个变体，我们都应该有；我们多出来的或少掉的，要能解释。
- **相关系数**：同样的软件版本、同样的参数、同样的原始数据，理论上应该**完全一样**（相关 = 1）。如果不是 1，要找出差在哪。

### 执行

In [ ]:
results = {}
for kind, title in (("abundance", "Abundance"), ("binding", "Binding")):
    fetch(f"{RESULTS}/psd95_pdz3_{kind}/fitness", DMS / f"results/{kind}/fitness")
    answer = f"GSE184042_fitness_PSD95-PDZ3_{title}_supp.txt.gz"
    s3.download_file(RAW, f"{READS}/{answer}", str(DMS / answer))
    ours = dms.read_fitness(DMS / f"results/{kind}/fitness/psd95_pdz3_{kind}")
    published = dms.read_published(DMS / answer)
    joined, summary = dms.compare(ours, published)
    results[kind] = joined
    print(f"--- {kind}")
    for k, v in summary.items():
        print(f"  {k:28s} {v:,.4f}" if isinstance(v, float) else f"  {k:28s} {v:,}")

### 结果怎么看

| | binding | abundance |
|---|---|---|
| 我们算出的变体数 | 8,148 | 6,457 |
| 作者公布的变体数 | 8,252 | 6,977 |
| 两边都有 | 7,901 | 6,291 |
| **相关系数** | **1.0000** | **1.0000** |
| 差异的中位数 | 9.1e-05 | 1.9e-04 |
| 差超过 0.01 的变体 | 10 / 7,901 | 10 / 6,291 |

**精确复现了。** 相关系数 1.0，差异的中位数是 0.00009——这是浮点运算的舍入级别。拟合一条直线，斜率 0.9998、截距 −0.0001。差超过 0.01 的只有 10 个变体，都是误差本来就大的双突变（sigma 0.14–0.15）。

两边变体数不完全一样（binding 我们多 247 个、少 351 个；abundance 多 166 个、少 686 个），这很正常：我们多出来的那些**噪声更大**（sigma 中位数 0.197，共有变体是 0.107），说明作者还做了一步我们没有的筛选；他们多出来的那些，是我们的最低输入计数门槛卡掉的。**重要的是：两边都有的 7,901 个，数值一致。**

这一条结果的意义：它证明**整条流程是对的**——去固定序列、拼接双端、计数、过滤、误差模型、算适应度，每一步都和原作者做的一样。接下来 14c 用这些数字拟合能量时，起点是可信的。

### 面试怎么说
> "I reprocessed a published ddPCA experiment from the raw FASTQ with the authors' own DiMSum version and settings, wrapped in Nextflow on AWS Batch, and reconciled variant by variant against their deposited fitness: Pearson 1.0000 on both phenotypes, over 7,901 and 6,291 shared variants, median absolute difference around 1e-04, and the handful of variants only one side has are the noisy ones."

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (kind, joined) in zip(axes, results.items()):
    for n, color in ((1, "#2a7ab9"), (2, "#e08b3a")):
        part = joined[joined.Nham_aa_published == n]
        ax.scatter(part.fitness_published, part.fitness_ours, s=4, alpha=0.4, color=color,
                   label=f"{'单' if n == 1 else '双'}突变 ({len(part):,})")
    lim = [joined[["fitness_published", "fitness_ours"]].min().min(), joined[["fitness_published", "fitness_ours"]].max().max()]
    ax.plot(lim, lim, color="grey", lw=0.8)
    ax.set(title=f"PSD95-PDZ3 {kind}", xlabel="作者公布的适应度（GEO）", ylabel="我们从读段算出的适应度")
    ax.legend(loc="upper left", markerscale=3)
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "DejaVu Sans"]
plt.tight_layout()

---

## 第 9 步 · 结果在哪，给 14c 用

### 在干什么
给这次运行的结果写一份"来历说明"（manifest），和结果放在一起。

### 背景
结果已经在 S3 了：流程的 `--outdir` 直接就是 `s3://<artifacts>/pipelines/dms_fitness/<实验>/`，Batch 上的机器跑完直接写进去，**从没经过你的电脑**。这是在云上跑流程的另一个好处：产出天然就在一个有版本、有权限、不会丢的地方。

还差一样：三个月后有人拿到这些适应度表，要能回答"**哪一版代码、哪一版容器、哪一份参数**跑出来的"。所以在旁边放一个 `manifest.json`。14c 读结果时先读它。

In [ ]:
commit = subprocess.run("git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run("git status --porcelain pipelines/", shell=True, capture_output=True, text=True).stdout.strip()
for kind in ("binding", "abundance"):
    prefix = f"{RESULTS}/psd95_pdz3_{kind}"
    files = sum(page.get("KeyCount", 0) for page in s3.get_paginator("list_objects_v2").paginate(Bucket=ARTIFACTS, Prefix=prefix))
    manifest = {"experiment": f"psd95_pdz3_{kind}", "pipeline": "pipelines/dms_fitness",
                "code_version": commit + ("-dirty" if dirty else ""), "executor": "awsbatch", "queue": QUEUE,
                "params_file": f"pipelines/dms_fitness/experiments/psd95_pdz3_{kind}.yaml",
                "containers": {"dimsum": "quay.io/biocontainers/r-dimsum:1.2.8--r41hdfd78af_0",
                               "fastqc": "quay.io/biocontainers/fastqc:0.12.1--hdfd78af_0",
                               "multiqc": "quay.io/biocontainers/multiqc:1.33--pyhdfd78af_0"},
                "reads": f"s3://{RAW}/{READS}/", "files": files}
    s3.put_object(Bucket=ARTIFACTS, Key=f"{prefix}/manifest.json", Body=json.dumps(manifest, indent=2).encode())
    print(f"{kind}: {files} 个文件 + manifest.json  →  s3://{ARTIFACTS}/{prefix}/")

---

## 这一章做完了什么

- 24 个原始 FASTQ（约 2.6 GB）和作者的 2 张适应度表进了 S3 raw，**MD5 和 ENA 逐个核对一致**
- 写了一个 **Nextflow 流程**：FastQC → MultiQC → DiMSum 计数 → 作者的双突变过滤 → DiMSum 适应度；每一步钉死容器版本（DiMSum 1.2.8，和论文一致），参数和代码分开，CI 里有 lint 和 stub 测试
- 用 Terraform 建了 **AWS Batch**（Spot、空闲时零台机器、最小权限、S3 工作目录 7 天自动清），流程换一个 profile 就从本机搬上云
- **质控**：要紧的三项（质量、N、接头）全部通过；4 项红灯是扩增子文库的固有特征
- 一路踩了六个坑并都写清楚了：内存不足被系统杀进程、磁盘写满、笔记本不适合 → 上云、账号的 vCPU 配额和 Free plan 机型限制、容器里的 AWS 命令行缺库、临时凭证中途过期
- **对账**：和作者 2022 年公布的适应度逐个变体比对，两个实验都是**相关系数 1.0000**（binding 7,901 个、abundance 6,291 个共有变体），差异中位数 1e-04 量级
- 结果直接写进 S3，旁边有 manifest 写明代码、容器和参数版本，14c 直接读

## 下一步

**14c · 一个数拆成两个能量**：手里现在有了同一个蛋白的**蛋白量**和**结合**两种测量（这一章的产出），加上 14a 的 kcal/mol 稳定性模型。14c 搭物理读数层：测蛋白量的实验只看折叠，测结合的实验看折叠 × 结合，从而把两个能量拆开。

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14b reads to fitness with Nextflow and DiMSum' && git log --oneline -1")